In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    USE_SNS = True
except ImportError:
    USE_SNS = False


# =========================================================
# 1) حساب MEDIAN من 30 ملف → 800 صف
# =========================================================
def merge_and_median(
    runs_dir=".",
    run_prefix="Run",
    run_suffix=".csv",
    n_runs=30,
    out_csv="Median_800_Experiments.csv"
):
    dfs = []

    for run_id in range(1, n_runs + 1):
        fname = os.path.join(runs_dir, f"{run_prefix}{run_id}{run_suffix}")
        if not os.path.isfile(fname):
            print(f"[merge] File missing: {fname}")
            continue

        df = pd.read_csv(fname)
        df["Run"] = run_id
        dfs.append(df)

    if not dfs:
        raise RuntimeError("No Run files loaded.")

    merged = pd.concat(dfs, ignore_index=True)
    print(f"[merge] Total rows loaded: {len(merged)}")

    keys = ["Dataset", "Drift Speed", "Labeling Strategy", "Imbalance", "Label"]
    numeric_cols = merged.select_dtypes(include="number").columns.tolist()

    remove_cols = ["Run", "stream_seed", "model_seed"] + keys
    numeric_cols = [c for c in numeric_cols if c not in remove_cols]

    median_df = (
        merged.groupby(keys)[numeric_cols]
        .median()
        .reset_index()
    )

    print(f"[median] Final rows: {len(median_df)} (expected ~800)")

    median_df.to_csv(out_csv, index=False)
    print(f"[median] Saved Median file: {out_csv}")

    return merged, median_df


# =========================================================
# 2) رسم GMean (10 صور فقط – كل Dataset بصورة واحدة)
# =========================================================
def plot_gmean_heatmaps_per_dataset(df, output_dir="GMean_Heatmaps", dpi=300):
    os.makedirs(output_dir, exist_ok=True)
    df = df.copy()
    df["Imbalance"] = df["Imbalance"].astype(int)
    df["Label"] = df["Label"].astype(int)

    datasets = sorted(df["Dataset"].unique())
    drift_levels = sorted(df["Drift Speed"].unique())
    strat_levels = sorted(df["Labeling Strategy"].unique())

    for ds in datasets:
        sub_ds = df[df["Dataset"] == ds]

        fig, axes = plt.subplots(
            nrows=2, ncols=2,
            figsize=(12, 8),
            squeeze=False
        )

        vmin = sub_ds["GMean"].min()
        vmax = sub_ds["GMean"].max()

        for i, drift in enumerate(drift_levels):
            for j, strat in enumerate(strat_levels):
                ax = axes[i][j]

                sub = sub_ds[
                    (sub_ds["Drift Speed"] == drift) &
                    (sub_ds["Labeling Strategy"] == strat)
                ]

                if sub.empty:
                    ax.axis("off")
                    continue

                pivot = sub.pivot_table(
                    index="Imbalance",
                    columns="Label",
                    values="GMean",
                    aggfunc="mean"
                )

                pivot = pivot.sort_index(axis=0).sort_index(axis=1)

                if USE_SNS:
                    sns.heatmap(
                        pivot, ax=ax,
                        annot=True, fmt=".3f",
                        cmap="viridis",
                        vmin=vmin, vmax=vmax,
                        cbar=False
                    )
                else:
                    im = ax.imshow(
                        pivot.values, aspect="auto",
                        vmin=vmin, vmax=vmax
                    )
                    for r in range(pivot.shape[0]):
                        for c in range(pivot.shape[1]):
                            ax.text(c, r, f"{pivot.values[r,c]:.2f}",
                                    ha="center", va="center")

                    ax.set_xticks(range(len(pivot.columns)))
                    ax.set_xticklabels(pivot.columns)
                    ax.set_yticks(range(len(pivot.index)))
                    ax.set_yticklabels(pivot.index)

                ax.set_title(f"{drift} – {strat}")
                ax.set_xlabel("Label (%)")
                ax.set_ylabel("Imbalance (%)")

        fig.suptitle(f"GMean – {ds}", fontsize=16)
        fig.tight_layout(rect=[0, 0, 1, 0.95])

        fname = os.path.join(output_dir, f"GMean_{ds}.png")
        fig.savefig(fname, dpi=dpi)
        plt.close(fig)

        print(f"[GMean-plot] Saved: {fname}")


# =========================================================
# 3) رسم Accuracy كـ line plots (10 صور فقط)
# =========================================================
def plot_accuracy_lines_per_dataset(df, output_dir="Accuracy_Plots", dpi=300):
    os.makedirs(output_dir, exist_ok=True)
    df = df.copy()
    df["Imbalance"] = df["Imbalance"].astype(int)
    df["Label"] = df["Label"].astype(int)

    datasets = sorted(df["Dataset"].unique())
    drift_levels = sorted(df["Drift Speed"].unique())
    strat_levels = sorted(df["Labeling Strategy"].unique())

    for ds in datasets:
        sub_ds = df[df["Dataset"] == ds]

        fig, axes = plt.subplots(
            nrows=2, ncols=2,
            figsize=(12, 8),
            squeeze=False
        )

        for i, drift in enumerate(drift_levels):
            for j, strat in enumerate(strat_levels):
                ax = axes[i][j]

                sub = sub_ds[
                    (sub_ds["Drift Speed"] == drift) &
                    (sub_ds["Labeling Strategy"] == strat)
                ]

                if sub.empty:
                    ax.axis("off")
                    continue

                for imb in sorted(sub["Imbalance"].unique()):
                    sub_imb = sub[sub["Imbalance"] == imb].sort_values("Label")

                    ax.plot(
                        sub_imb["Label"],
                        sub_imb["Accuracy"],
                        marker="o",
                        linestyle="-",
                        label=f"Imb={imb}%"
                    )

                ax.set_title(f"{drift} – {strat}")
                ax.set_ylim(0, 1)
                ax.set_xlabel("Label (%)")
                ax.set_ylabel("Accuracy")
                ax.grid(True, linestyle="--", alpha=0.5)
                ax.legend(fontsize=8, loc="lower right")

        fig.suptitle(f"Accuracy – {ds}", fontsize=16)
        fig.tight_layout(rect=[0, 0, 1, 0.95])

        fname = os.path.join(output_dir, f"Accuracy_{ds}.png")
        fig.savefig(fname, dpi=dpi)
        plt.close(fig)

        print(f"[Acc-plot] Saved: {fname}")


# =========================================================
# 4) نقطة التنفيذ الرئيسية
# =========================================================
if __name__ == "__main__":
    model_name = "OSNN"   # ضع اسم النموذج هنا يدوياً

    runs_dir = "."
    median_csv = f"Median_800_Experiments_{model_name}.csv"

    merged_df, median_df = merge_and_median(
        runs_dir=runs_dir,
        out_csv=median_csv
    )

    plot_gmean_heatmaps_per_dataset(
        median_df,
        output_dir=f"GMean_Heatmaps_{model_name}"
    )

    plot_accuracy_lines_per_dataset(
        median_df,
        output_dir=f"Accuracy_Plots_{model_name}"
    )

    print("\n=== DONE COMPLETELY ===\n")



[merge] Total rows loaded: 24000
[median] Final rows: 800 (expected ~800)
[median] Saved Median file: Median_800_Experiments_OSNN.csv
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_AGRAWAL1.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_AGRAWAL2.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_AGRAWAL3.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_AGRAWAL4.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_SEA1.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_SEA2.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_SINE1.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_SINE2.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_STAGGER1.png
[GMean-plot] Saved: GMean_Heatmaps_OSNN\GMean_STAGGER2.png
[Acc-plot] Saved: Accuracy_Plots_OSNN\Accuracy_AGRAWAL1.png
[Acc-plot] Saved: Accuracy_Plots_OSNN\Accuracy_AGRAWAL2.png
[Acc-plot] Saved: Accuracy_Plots_OSNN\Accuracy_AGRAWAL3.png
[Acc-plot] Saved: Accuracy_Plots_OSNN\Accuracy_AGRAWAL4.png
[Acc-plot] Saved: Accuracy_Plots_OSNN\Accuracy_SEA